# Libraries

In [9]:
!pip install -q bertopic sentence-transformers

In [10]:
import pandas as pd
import numpy as np
import re
import html
import os
import glob
# for the sentence transformer
from sentence_transformers import SentenceTransformer
# Import BERTopic for topic modeling
from bertopic import BERTopic

# Loading the dataset

In [11]:
import os
import pandas as pd

# 1. Inspect what is actually inside /kaggle/input
found_files = {}
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file in ["both_train.csv", "both_val.csv", "both_test.csv"]:
            found_files[file] = os.path.join(root, file)

print("Found files:", found_files)

# 2. Check that all files exist
if "both_train.csv" not in found_files:
    raise FileNotFoundError("Files not detected under /kaggle/input. Did the dataset upload finish and is it attached to this notebook?")

# 3. Load cleanly
train_df = pd.read_csv(found_files["both_train.csv"])
val_df = pd.read_csv(found_files["both_val.csv"])
test_df = pd.read_csv(found_files["both_test.csv"])

print("\nLoaded successfully!")
print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Found files: {'both_val.csv': '/kaggle/input/datasets/namitharoystat/mhc-tc-datasets/both_val.csv', 'both_train.csv': '/kaggle/input/datasets/namitharoystat/mhc-tc-datasets/both_train.csv', 'both_test.csv': '/kaggle/input/datasets/namitharoystat/mhc-tc-datasets/both_test.csv'}

Loaded successfully!
Train shape: (13727, 5)
Validation shape: (1488, 5)
Test shape: (1488, 5)


# Data Preprocessing

In [12]:
train_df["text"] = train_df["title"] + " " + train_df["post"]
val_df["text"] = val_df["title"] + " " + val_df["post"]
test_df["text"] = test_df["title"] + " " + test_df["post"]

In [13]:
train_df[["title", "post", "text"]].head()

,title,post,text
0,a question about the third conditional.,i was making questions for my students and i r...,a question about the third conditional. i was ...
1,the epitome of my life,i've recently requested testing accommodations...,the epitome of my life i've recently requested...
2,what are your favourites offbeat destinations ...,**cambodia** * koh rong: amazing beaches and a...,what are your favourites offbeat destinations ...
3,synesthesia survey (what colour is each month ...,synesthesia. what is synesthesia? according to...,synesthesia survey (what colour is each month ...
4,"science ama series: i’m phil baran, and i’m he...",i’m phil baran and i teach organic chemistry a...,"science ama series: i’m phil baran, and i’m he..."


## removing reduntant columns

In [14]:
train_df = train_df.drop(columns=["ID", "class_name", "title", "post"])
val_df = val_df.drop(columns=["ID", "class_name", "title", "post"])
test_df = test_df.drop(columns=["ID", "class_name", "title", "post"])


## Remove HTML tags / markup

In [15]:
print(train_df["text"].iloc[0])

a question about the third conditional. i was making questions for my students and i ran into a little tricky grammar. which is correct (focusing on the latter part of the sentence): &amp;#x200b; * if you had traveled to australia yesterday, what wouldn't you have done while you had been there? * if you had traveled to australia yesterday, what wouldn't you have done while you were there? &amp;#x200b; the second \*feels\* right, but i want to make sure.


In [16]:
def clean_text(text):
    # Removing HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # Remove Reddit-style markdown formatting
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)  # [text](url)

    return text

In [17]:
# applying the clean text function on all 3 data frames
train_df["text"] = train_df["text"].apply(clean_text)
val_df["text"] = val_df["text"].apply(clean_text)
test_df["text"] = test_df["text"].apply(clean_text)

## Removing html entities

In [18]:
train_df["text"] = train_df["text"].apply(html.unescape)
val_df["text"] = val_df["text"].apply(html.unescape)
test_df["text"] = test_df["text"].apply(html.unescape)

## Replacing Reddit usernames

In [19]:
train_df["text"] = train_df["text"].apply(
    lambda x: re.sub(r'u/\w+', 'USER', x)
)

val_df["text"] = val_df["text"].apply(
    lambda x: re.sub(r'u/\w+', 'USER', x)
)

test_df["text"] = test_df["text"].apply(
    lambda x: re.sub(r'u/\w+', 'USER', x)
)

## Replacing mentions to @user

In [20]:
train_df["text"] = train_df["text"].apply(
    lambda x: re.sub(r'@\w+', '@USER', x)
)

val_df["text"] = val_df["text"].apply(
    lambda x: re.sub(r'@\w+', '@USER', x)
)

test_df["text"] = test_df["text"].apply(
    lambda x: re.sub(r'@\w+', '@USER', x)
)

## Removing extra whitespace

In [21]:
train_df["text"] = train_df["text"].apply(
    lambda x: re.sub(r'\s+', ' ', x).strip()
)

val_df["text"] = val_df["text"].apply(
    lambda x: re.sub(r'\s+', ' ', x).strip()
)

test_df["text"] = test_df["text"].apply(
    lambda x: re.sub(r'\s+', ' ', x).strip()
)

# Prepare text for analyses

In [22]:
class_labels = {
    0: "ADHD",
    1: "Anxiety",
    2: "Bipolar",
    3: "Depression",
    4: "PTSD",
    5: "Others"
}

In [23]:
class_texts = {
    class_labels[class_id]: train_df.loc[
        train_df["class_id"] == class_id, "text"
    ].tolist()
    for class_id in sorted(train_df["class_id"].unique())
}

all_texts = train_df["text"].tolist()

print("Total documents:", len(all_texts))

for class_name, texts in class_texts.items():
    print(class_name, ":", len(texts))

Total documents: 13727
ADHD : 2465
Anxiety : 2422
Bipolar : 2407
Depression : 2450
PTSD : 2001
Others : 1982


# Generate embeddings

In [24]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [25]:
# embedding for combined dataset
all_embeddings = embedding_model.encode(
    all_texts,
    show_progress_bar=True
)

Batches:   0%|          | 0/429 [00:00<?, ?it/s]

# Topic detection

## Without masking

In [26]:
# # Import BERTopic for topic modeling
# from bertopic import BERTopic

# # Initialize the BERTopic model
# # The sentence-transformer model is used to generate semantic representations
# # verbose=True displays the progress of the modeling process
# topic_model_all = BERTopic(
#     embedding_model=embedding_model,
#     verbose=True
# )

In [27]:
# # Fit BERTopic on all training texts
# # The pre-generated sentence embeddings are supplied to avoid recomputing them
# # topics_all contains the topic assigned to each text
# # probabilities_all contains the probability distribution of topic assignments
# topics_all, probabilities_all = topic_model_all.fit_transform(
#     all_texts,
#     embeddings=all_embeddings
# )

2026-09-23 07:27:37,976 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-09-23 07:28:12,280 - BERTopic - Dimensionality - Completed ✓
2026-09-23 07:28:12,282 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-23 07:28:13,024 - BERTopic - Cluster - Completed ✓
2026-09-23 07:28:13,031 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-23 07:28:15,215 - BERTopic - Representation - Completed ✓


In [28]:
# # getting model info
# topic_model_all.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,5920,-1_my_to_and_it,"[my, to, and, it, me, the, that, of, in, have]","[urgent: i suspect i have ptsd, but i have no ..."
1,0,390,0_adhd_you_that_it,"[adhd, you, that, it, to, they, and, with, of,...",['you only want to be labelled adhd so you hav...
2,1,318,1_depression_depressed_you_fuck,"[depression, depressed, you, fuck, feel, it, j...","[""i wish he talked about it more."" yeah, becau..."
3,2,309,2_school_class_college_my,"[school, class, college, my, semester, to, and...","[symptoms finally in remission (knock on wood,..."
4,3,304,3_friends_people_feel_alone,"[friends, people, feel, alone, they, me, lonel...",[i'm 20 don't have any friends and never had a...
...,...,...,...,...,...
122,121,11,121_bookstore_tomorrow_anxiety_proud,"[bookstore, tomorrow, anxiety, proud, today, g...",[sunday nights are the worst i have ptsd and l...
123,122,10,122_term_bored_brain_my,"[term, bored, brain, my, and, to, always, in, ...","[all in or all out, adhd trait? anyone else st..."
124,123,10,123_subtitles_movies_shows_watching,"[subtitles, movies, shows, watching, series, w...",[watching a series with or without subtitles? ...
125,124,10,124_sub_adhd_subscribed_posts,"[sub, adhd, subscribed, posts, subs, capslock,...",[i just found you people! i’ve felt so alone i...


In [29]:
# # creating complete topic table
# topic_info_all = topic_model_all.get_topic_info()

# topic_info_all

,Topic,Count,Name,Representation,Representative_Docs
0,-1,5920,-1_my_to_and_it,"[my, to, and, it, me, the, that, of, in, have]","[urgent: i suspect i have ptsd, but i have no ..."
1,0,390,0_adhd_you_that_it,"[adhd, you, that, it, to, they, and, with, of,...",['you only want to be labelled adhd so you hav...
2,1,318,1_depression_depressed_you_fuck,"[depression, depressed, you, fuck, feel, it, j...","[""i wish he talked about it more."" yeah, becau..."
3,2,309,2_school_class_college_my,"[school, class, college, my, semester, to, and...","[symptoms finally in remission (knock on wood,..."
4,3,304,3_friends_people_feel_alone,"[friends, people, feel, alone, they, me, lonel...",[i'm 20 don't have any friends and never had a...
...,...,...,...,...,...
122,121,11,121_bookstore_tomorrow_anxiety_proud,"[bookstore, tomorrow, anxiety, proud, today, g...",[sunday nights are the worst i have ptsd and l...
123,122,10,122_term_bored_brain_my,"[term, bored, brain, my, and, to, always, in, ...","[all in or all out, adhd trait? anyone else st..."
124,123,10,123_subtitles_movies_shows_watching,"[subtitles, movies, shows, watching, series, w...",[watching a series with or without subtitles? ...
125,124,10,124_sub_adhd_subscribed_posts,"[sub, adhd, subscribed, posts, subs, capslock,...",[i just found you people! i’ve felt so alone i...


In [30]:
# # Convert the word lists and representative documents to text
# topic_info_excel = topic_info_all.copy()

# topic_info_excel["Representation"] = topic_info_excel["Representation"].apply(
#     lambda x: ", ".join(x) if isinstance(x, list) else x
# )

# topic_info_excel["Representative_Docs"] = topic_info_excel["Representative_Docs"].apply(
#     lambda x: " || ".join(x) if isinstance(x, list) else x
# )

In [31]:
# # Export all of them to Excel
# topic_info_excel.to_excel(
#     "BERTopic_All_Topics.xlsx",
#     index=False
# )
# topic_info_excel["Interpreted_Theme"] = ""
# #topic_info_excel.to_excel("BERTopic_All_Topics.xlsx",index=False)

Note: Masking Condition Names Before Theme Detection

The initial BERTopic analysis was performed on the complete dataset without masking the mental health condition names. The resulting topic table showed that several topics were strongly dominated by explicit condition names such as ADHD, bipolar, and anxiety.

This indicates that BERTopic may be identifying the condition labels themselves as topics, rather than discovering the underlying themes or experiences expressed in the text.

Therefore, for the disease-wise theme detection, the condition names should be masked before applying BERTopic. The condition names will be replaced with a neutral placeholder such as DISEASE.

This approach will reduce the influence of explicit condition labels and encourage BERTopic to identify underlying linguistic, behavioural, emotional, and experiential themes within each disease-specific group.

Planned approach:

Cleaned text → Mask condition names → Separate texts by condition → BERTopic → Identify underlying themes

This masking will be applied separately to the six classes before conducting the disease-wise topic analysis.

## With Masking

In [ ]:
def mask_conditions(text):
    patterns = [
        r'\battention deficit hyperactivity disorder\b',
        r'\badhd\b',
        r'\banxiety disorder\b',
        r'\banxiety\b',
        r'\bbipolar disorder\b',
        r'\bbipolar\b',
        r'\bdepressive disorder\b',
        r'\bdepression\b',
        r'\bpost[- ]traumatic stress disorder\b',
        r'\bptsd\b',
        r'\bc[- ]?ptsd\b'
    ]
    
    for pattern in patterns:
        text = re.sub(pattern, 'DISEASE', text, flags=re.IGNORECASE)
    
    return text

In [33]:
train_df["text_masked"] = train_df["text"].apply(mask_conditions)
val_df["text_masked"] = val_df["text"].apply(mask_conditions)
test_df["text_masked"] = test_df["text"].apply(mask_conditions)

In [34]:
# checking how the masking happens
condition_names = [
    "adhd",
    "anxiety",
    "bipolar",
    "depression",
    "ptsd"
]

for condition in condition_names:
    count = train_df["text_masked"].str.lower().str.count(condition).sum()
    print(condition, ":", count)

adhd : 60
anxiety : 5
bipolar : 55
depression : 18
ptsd : 4


In [35]:
for condition in ["adhd", "bipolar"]:
    pattern = rf'\b{condition}\b'
    
    rows = train_df[
        train_df["text_masked"].str.contains(
            pattern,
            case=False,
            regex=True,
            na=False
        )
    ]
    
    print(f"\n{condition}: {len(rows)} remaining rows")
    print(rows["text_masked"].head(10).tolist())


adhd: 0 remaining rows
[]

bipolar: 0 remaining rows
[]


In [36]:
print(train_df["text"].iloc[15])
print("\n--- MASKED ---")
print(train_df["text_masked"].iloc[15])

i wish people would understand i wish people would understand that depression isn't always crying in the shower and playing sad songs in bed. sometimes it's not wanting to talk to anyone for days and other times it's desperately needing to be around people. sometimes it's having no appetite even though you haven't eaten anything since yesterday and sometimes it's eating everything you have in the fridge. depression isn't your boyfriend/girlfriend holding you and telling you that everything is going to be okay. it's sitting across the table, not eating, having them ask you what's wrong and knowing that you're ruining their night because you can't seem to snap out of it and just be happy. it's the frustrating feeling of desperately wanting to enjoy something and just fucking be normal for once. it's keeping things a secret from the people you love because you don't want them to look at you differently. no, depression isn't beautiful black and white images. depression is lonely and frustr

In [37]:
for condition in condition_names:
    rows = train_df[
        train_df["text_masked"].str.contains(
            condition, case=False, regex=False, na=False
        )
    ]
    
    print(f"\n{condition}: {len(rows)} remaining rows")
    
    if len(rows) > 0:
        print(rows["text_masked"].head(3).tolist())


adhd: 53 remaining rows
['how have you improved your social skills? all my life i’ve noticed that it’s hard for me to put time and effort into friendships, but sometimes i’ll even fixate on one person i guess until i move on to someone else. i never did this on purpose or to be hurtful, if anything i’ve usually noticed people move on from me if that happens. this didn’t really hit me until this past year when one of my friends called me out on this and even if it hurt at first i realized they were right. i also find it hard to have the motivation to see if i want to hang out with some friends and i often as myself if it’s worth the trouble or DISEASE to try and do so. i just always worry that my friends might find me too overbearing or annoying since sometimes when i’m too happy to be around someone, i do get hyper. i guess my point/question is how am i able to find a happy medium with friendships and social skills that seems to come naturally for nts? it seems like they don’t have as

In [38]:
for condition in condition_names:
    pattern = rf'\b{condition}\b'
    
    count = train_df["text_masked"].str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    ).sum()
    
    print(condition, ":", count)

adhd : 0
anxiety : 0
bipolar : 0
depression : 0
ptsd : 0


In [39]:
# preparing for topic detction
masked_texts = train_df["text_masked"].tolist()
print("done")

done


In [40]:
topic_model_masked = BERTopic(
    embedding_model=embedding_model
)

In [42]:
# Applying BERTopic to the masked text to identify underlying themes
# without allowing explicit mental health condition names to dominate the topics

topic_model_masked = BERTopic()

topics_masked, probs_masked = topic_model_masked.fit_transform(
    masked_texts
)
print("completed")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [43]:
# getting compete topic list
topic_info_masked = topic_model_masked.get_topic_info()

topic_info_masked

,Topic,Count,Name,Representation,Representative_Docs
0,-1,5681,-1_my_and_to_me,"[my, and, to, me, it, that, im, the, of, was]",[i am too scared to live my life how i want to...
1,0,438,0_disease_you_people_is,"[disease, you, people, is, mental, that, your,...",[surprisingly decent write-up on 21 of the rea...
2,1,437,1_do_time_doing_to,"[do, time, doing, to, work, things, something,...","[constant ""running out of time"" feeling for th..."
3,2,410,2_friends_people_feel_me,"[friends, people, feel, me, im, alone, dont, l...",[im afrad and anxious that i might never find ...
4,3,388,3_suicide_kill_suicidal_myself,"[suicide, kill, suicidal, myself, die, want, d...",[does anyone else want something bad to happen...
...,...,...,...,...,...
130,129,11,129_delusions_delusion_person_they,"[delusions, delusion, person, they, maybe, par...",[i was convinced the government was spying on ...
131,130,11,130_fog_brain_dissociation_brainhead,"[fog, brain, dissociation, brainhead, quetiapi...",[how to deal with brain fog as a student does ...
132,131,11,131_fucking_what_be_im,"[fucking, what, be, im, is, ive, point, me, ge...","[crippling DISEASE hello everyone, long story ..."
133,132,11,132_embarrassing_cringing_stupid_cringe,"[embarrassing, cringing, stupid, cringe, memor...",[why am i constantly cringing at my past self?...


In [48]:
# Export the complete topic information to an Excel file

topic_info_masked.to_excel(
    "BERTopic_Masked_All_Topics.xlsx",
    index=False
)

In [49]:
# Convert list columns to readable text before exporting

topic_info_masked_excel = topic_info_masked.copy()

topic_info_masked_excel["Representation"] = (
    topic_info_masked_excel["Representation"]
    .apply(lambda x: ", ".join(x) if isinstance(x, list) else x)
)

topic_info_masked_excel["Representative_Docs"] = (
    topic_info_masked_excel["Representative_Docs"]
    .apply(lambda x: " || ".join(x) if isinstance(x, list) else x)
)

#topic_info_masked_excel.to_excel(
    "BERTopic_Masked_All_Topics.xlsx",
    index=False
)

In [50]:
# Reduce the fine-grained topics down to a target number.
# BERTopic re-clusters existing topic embeddings hierarchically,
# so this doesn't require re-running the whole pipeline from scratch.

topic_model_all.reduce_topics(all_texts, nr_topics=6)

# Check the result
print(topic_model_all.get_topic_info())

   Topic  Count                    Name  \
0     -1   5920        -1_to_and_the_my   
1      0   6327         0_to_and_the_it   
2      1    861         1_the_of_to_and   
3      2    245         2_the_to_and_of   
4      3    228        3_the_of_math_to   
5      4    146  4_english_x200b_to_the   

                                      Representation  \
0      [to, and, the, my, it, of, that, me, in, you]   
1      [to, and, the, it, my, of, that, in, me, you]   
2   [the, of, to, and, in, user, trump, for, is, we]   
3  [the, to, and, of, it, you, album, that, for, in]   
4  [the, of, math, to, is, in, and, mathematics, ...   
5  [english, x200b, to, the, you, it, in, is, nat...   

                                 Representative_Docs  
0  [screwed up my life beyond belief. what are my...  
1  [the big helpful post of strategies that i cur...  
2  [how come bernie needs a plan to pay for his h...  
3  [the best ones always die this way. a couple w...  
4  [second year math major loo

In [51]:
from sklearn.feature_extraction.text import CountVectorizer

# Build a stopword list: standard English stopwords, plus a few things
# specific to this cleaned dataset that would otherwise pollute every topic:
# - "disease" is your masking token, so it appears in almost every post
#   regardless of theme, and tells you nothing
# - "x200b" is a Reddit zero-width-space artifact left over from cleaning
# - "user" is your anonymization placeholder for usernames/mentions
# - contractions without apostrophes ("im", "dont", "youre") since your
#   cleaning step likely stripped apostrophes before this point
custom_stopwords = ["disease", "x200b", "user", "im", "ive", "dont",
                     "youre", "theyre", "hes", "shes", "its", "thats"]

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 1)
)
# Extend sklearn's built-in English stopword list with your custom ones
vectorizer_model.stop_words = list(vectorizer_model.get_stop_words()) + custom_stopwords if False else None

# Simpler and more reliable: pass a combined stopword list directly
from sklearn.feature_extraction import text as sk_text
all_stopwords = list(sk_text.ENGLISH_STOP_WORDS.union(custom_stopwords))

vectorizer_model = CountVectorizer(stop_words=all_stopwords)

# Recompute topic representations using the better vectorizer —
# this reuses your existing topic assignments, it does NOT re-embed or re-cluster
topic_model_all.update_topics(all_texts, vectorizer_model=vectorizer_model)

print(topic_model_all.get_topic_info())

   Topic  Count                                 Name  \
0     -1   5920                -1_just_like_feel_don   
1      0   6327                 0_just_like_feel_don   
2      1    861             1_trump_com_science_data   
3      2    245              2_album_music_song_band   
4      3    228  3_math_mathematics_calculus_algebra   
5      4    146      4_english_native_language_words   

                                      Representation  \
0  [just, like, feel, don, ve, know, time, life, ...   
1  [just, like, feel, don, people, time, ve, know...   
2  [trump, com, science, data, questions, house, ...   
3  [album, music, song, band, songs, playlist, re...   
4  [math, mathematics, calculus, algebra, number,...   
5  [english, native, language, words, uk, speaker...   

                                 Representative_Docs  
0  [screwed up my life beyond belief. what are my...  
1  [the big helpful post of strategies that i cur...  
2  [how come bernie needs a plan to pay for his h

In [52]:
# Restrict theme discovery to the five real conditions (exclude class 5 / "None")
condition_df = train_df[train_df["class_id"] != 5].reset_index(drop=True)
condition_texts = condition_df["text"].tolist()

# Re-generate embeddings on this smaller, focused set
condition_embeddings = embedding_model.encode(
    condition_texts,
    show_progress_bar=True
)

# Fresh BERTopic run on just the condition-related posts
topic_model_conditions = BERTopic(
    embedding_model=embedding_model,
    verbose=True
)
topics_cond, probs_cond = topic_model_conditions.fit_transform(
    condition_texts, condition_embeddings
)

# Same stopword cleanup as before
topic_model_conditions.update_topics(condition_texts, vectorizer_model=vectorizer_model)

# Reduce to 5 (one per condition) or 6 if you want room for a mixed/general theme
topic_model_conditions.reduce_topics(condition_texts, nr_topics=5)

print(topic_model_conditions.get_topic_info())

Batches:   0%|          | 0/368 [00:00<?, ?it/s]

2026-09-23 08:18:21,904 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-09-23 08:18:27,462 - BERTopic - Dimensionality - Completed ✓
2026-09-23 08:18:27,464 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-23 08:18:28,103 - BERTopic - Cluster - Completed ✓
2026-09-23 08:18:28,110 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-23 08:18:29,943 - BERTopic - Representation - Completed ✓
2026-09-23 08:18:32,839 - BERTopic - Topic reduction - Reducing number of topics
2026-09-23 08:18:32,869 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-23 08:18:34,518 - BERTopic - Representation - Completed ✓
2026-09-23 08:18:34,523 - BERTopic - Topic reduction - Reduced number of topics from 106 to 5


   Topic  Count                          Name  \
0     -1   5430         -1_just_like_feel_don   
1      0   5795          0_just_like_feel_don   
2      1    396     1_birthday_just_like_feel   
3      2     93       2_girl_just_really_said   
4      3     31  3_weighted_jaw_blanket_teeth   

                                      Representation  \
0  [just, like, feel, don, ve, know, life, time, ...   
1  [just, like, feel, don, people, time, ve, adhd...   
2  [birthday, just, like, feel, day, today, peopl...   
3  [girl, just, really, said, like, asked, class,...   
4  [weighted, jaw, blanket, teeth, blankets, butt...   

                                 Representative_Docs  
0  [i can't hold down a job because of panic atta...  
1  [how do you cope? (new to reddit, don't know a...  
2  [i hate my birthday this probably isn’t a very...  
3  [no one has to read this, it’s just one of tho...  
4  [weighted blankets my doctor recommended a wei...  


In [1]:
# topic_info_excel = topic_info_df.drop(
#     columns=["Representative_Docs"],
#     errors="ignore"
# )

# topic_info_excel.to_excel(
#     "BERTopic_Conditions_Topics.xlsx",
#     index=False
# )

# print("Saved:", topic_info_excel.shape[0], "topics")

NameError: name 'topic_info_df' is not defined